# 09 · experiment/image — DL × privacy benchmark on dermoscopy (DermaMNIST melanoma)

Track notebook for the private-DL arm of `experiment/image`. Task: binary melanoma-vs-rest
on 28×28 RGB (10 pseudo-clinics, Dirichlet α=0.5 label skew, prevalence 0.1111; payload
provenance in `data/external/SOURCES.md`.). Model: CNN-Small (28,577 params, wire =
trainable params only). Transport: FedProx μ=0.1 (measured parity-oscillation fix vs FedAvg
on Dirichlet-skew pixels).

Four paradigms measured: **P1** per-image record-level DP-SGD with per-cell loss-threshold
MIA audit (attack AUC + TPR@FPR=1%); **P2** SecAgg+ analytic cost at CNN wire sizes plus the
cross-referenced TS deployment probe; **P3** FedCT consensus (10 isolated teachers, 512-row
train-carve public pool); **P4** verified-hybrid DDG integer-masked sums with norm proofs.
The matrix rows come from `results/dl_image_matrix.json`; payloads stay local.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
matrix = pd.DataFrame(json.loads((ROOT / 'results/dl_image_matrix.json').read_text())['rows'])
print(matrix.groupby('paradigm').size().rename('rows').to_string())

paradigm
P1                   19
P1-layerwise          7
P1-user-level         1
P2-cost              64
P2-measured           1
P3                   12
P3-clean-ceiling      6
P3-rr                 5
P3-rr-replication     1
P4                    7


In [2]:
p1 = matrix[matrix.paradigm == 'P1'].copy()
p1['miaAUC'] = p1['aux'].map(lambda a: a['mia_attack_auc'])
p1['tpr@1pct'] = p1['aux'].map(lambda a: a['mia_tpr_at_fpr1pct'])
piv = p1.pivot_table(index='epsilon_target', values=['metric_value', 'metric_worst', 'miaAUC'],
                     aggfunc=['mean', 'min', 'max']).round(3)
print('P1 record-level DP-SGD on CNN-S (FedProx mu=0.1): AUROC across seeds + MIA audit per cell')
piv

P1 record-level DP-SGD on CNN-S (FedProx mu=0.1): AUROC across seeds + MIA audit per cell


mean                              min               \
               metric_value metric_worst miaAUC metric_value metric_worst   
epsilon_target                                                              
0.5                   0.657        0.557  0.499        0.657        0.557   
1.0                   0.622        0.401  0.507        0.579        0.000   
2.0                   0.657        0.555  0.499        0.657        0.555   
4.0                   0.624        0.404  0.507        0.583        0.000   
8.0                   0.657        0.555  0.499        0.657        0.555   
16.0                  0.657        0.555  0.499        0.657        0.555   

                               max                      
               miaAUC metric_value metric_worst miaAUC  
epsilon_target                                          
0.5             0.499        0.657        0.557  0.499  
1.0             0.499        0.657        0.557  0.512  
2.0             0.499        0.657        0.555  0.499  
4.0             0.499        0.657        0.555  0.512  
8.0             0.499        0.657        0.555  0.499  
16.0            0.499        0.657        0.555  0.499

In [3]:
s42 = p1[p1['seed'] == 42].sort_values('epsilon_target', na_position='first')
print('Seed-42 full grid with MIA:')
s42[['epsilon_target', 'metric_value', 'metric_worst', 'miaAUC', 'tpr@1pct']].round(4)

Seed-42 full grid with MIA:


,epsilon_target,metric_value,metric_worst,miaAUC,tpr@1pct
0,NaN,0.6302,0.5209,0.4999,0.0080
1,0.5,0.6566,0.5571,0.4988,0.0075
2,1.0,0.6569,0.5571,0.4989,0.0070
3,2.0,0.6570,0.5549,0.4991,0.0070
4,4.0,0.6569,0.5549,0.4991,0.0075
5,8.0,0.6567,0.5549,0.4992,0.0075
6,16.0,0.6565,0.5549,0.4992,0.0075


In [4]:
p3 = matrix[matrix.paradigm == 'P3'].copy()
p3['sens'] = p3['aux'].map(lambda a: a['sensitivity'])
p3['sigv'] = p3['aux'].map(lambda a: round(a['sigma_votes']))
ceiling = matrix[matrix.paradigm == 'P3-clean-ceiling'][['metric_value', 'metric_worst']].iloc[0]
print('P3 FedCT — clean distilled student AUROC %.3f (worst %.3f); all paid cells NaN:'
      % (ceiling['metric_value'], ceiling['metric_worst']))
p3[['epsilon_target', 'sens', 'sigv', 'metric_value']].sort_values(['epsilon_target', 'sens'])

P3 FedCT — clean distilled student AUROC 0.595 (worst 0.499); all paid cells NaN:


,epsilon_target,sens,sigv,metric_value
29,0.5,conservative,27457,0.239170
30,0.5,refined,6128,0.393962
32,1.0,conservative,13729,0.239170
33,1.0,refined,3064,0.393962
35,2.0,conservative,6864,0.239170
36,2.0,refined,1532,0.393962
38,4.0,conservative,3432,0.239170
39,4.0,refined,766,0.405907
41,8.0,conservative,1716,0.239170
42,8.0,refined,383,0.425123


In [5]:
p4 = matrix[matrix.paradigm == 'P4'].copy()
p4['proofs'] = p4['aux'].map(lambda a: a['norm_proofs_ok'])
p4['kls'] = p4['aux'].map(lambda a: a['kls_feasible'])
print('P4 verified-hybrid DDG (mod-2^32 ring, scale 1e-3), seed 42:')
p4[['epsilon_target', 'metric_value', 'metric_worst', 'proofs', 'kls']].round(3)

P4 verified-hybrid DDG (mod-2^32 ring, scale 1e-3), seed 42:


,epsilon_target,metric_value,metric_worst,proofs,kls
50,NaN,0.649,0.549,True,False
51,0.5,0.628,0.504,True,True
52,1.0,0.638,0.525,True,True
53,2.0,0.628,0.514,True,True
54,4.0,0.627,0.511,True,True
55,8.0,0.627,0.513,True,True
56,16.0,0.632,0.515,True,True


In [6]:
lim = matrix[matrix.paradigm == 'P1-user-level']['aux'].iloc[0]
print('Patient-level DP status:', lim['status']); print(lim['reason'])
cost = matrix[matrix.paradigm == 'P2-cost']
print('\nP2 analytic rows:', len(cost), 'protocols:', sorted(cost['transport'].unique()))
mr = matrix[matrix.paradigm == 'P2-measured']['aux'].iloc[0]
print('Measured row:', mr['verdict'][:160])

Patient-level DP status: not-runnable
HAM10000 patient-key reattachment unrecoverable from the MedMNIST payload (undisclosed row permutation; ordering sweep found no match) - see README section 8

P2 analytic rows: 64 protocols: ['fastsecagg', 'lightsecagg', 'secagg', 'secagg_plus']
Measured row: protocol-construct verified on the TS probe; image track carries the analytic table at image wire sizes; end-to-end E2E runtime measurement at image sizes recor


In [7]:
p1l = matrix[matrix.paradigm == 'P1-layerwise'].copy()
p1l['mia'] = p1l['aux'].map(lambda a: a['mia_attack_auc'])
piv = p1l.pivot_table(index='epsilon_target', values=['metric_value', 'metric_worst', 'mia'],
                      aggfunc='mean').round(3)
print('P1 PER-LAYER clip arm (5 module groups; noise sigma*sqrt(5)*C per layer; same composed eps):')
print(piv)
print()
cf = p1l.iloc[len(p1l)//2]['aux']['clip_frac_per_layer_final']
md = p1l.iloc[len(p1l)//2]['aux']['med_norm_per_layer_final']
print('Filter health (final round; example cell): per-layer clip fraction | pre-clip median norm')
for nm in cf:
    print(f'  {nm:<10} clip={cf[nm]:.3f}  med||g||={md[nm]:.3f}')

P1 PER-LAYER clip arm (5 module groups; noise sigma*sqrt(5)*C per layer; same composed eps):
                metric_value  metric_worst    mia
epsilon_target                                   
0.5                    0.642         0.554  0.499
1.0                    0.668         0.572  0.498
2.0                    0.666         0.571  0.499
4.0                    0.663         0.568  0.499
8.0                    0.664         0.568  0.499
16.0                   0.664         0.568  0.499

Filter health (final round; example cell): per-layer clip fraction | pre-clip median norm
  conv.0     clip=0.160  med||g||=0.156
  conv.3     clip=0.160  med||g||=0.153
  conv.6     clip=0.160  med||g||=0.166
  head.1     clip=0.160  med||g||=0.158
  head.3     clip=0.160  med||g||=0.155


In [8]:
rr = matrix[matrix.paradigm == 'P3-rr'].copy()
rep = matrix[matrix.paradigm == 'P3-rr-replication']['aux'].iloc[0]
print('P3-RR: local randomized-response votes (advanced comp, delta=1e-5), q=512, K=10:')
t = rr[['epsilon_target', 'metric_value']].copy()
t['eps_vote'] = rr['aux'].map(lambda a: round(a['eps_vote'], 5))
t['p_flip'] = rr['aux'].map(lambda a: round(a['p_flip'], 4))
t['sigma_equiv'] = rr['aux'].map(lambda a: round(a['sigma_equiv'], 1))
t['pos_rr/clean'] = rr['aux'].map(lambda a: f"{a['pos_count_rr']}/{a['pos_count_clean']}")
t['pos_recall'] = rr['aux'].map(lambda a: round(a['pos_recall_vs_clean'], 2))
print(t.sort_values('epsilon_target').to_string(index=False))
print()
print('rerun-fidelity replication (teachers retrained once, gaussian cells vs committed grid):', rep)

P3-RR: local randomized-response votes (advanced comp, delta=1e-5), q=512, K=10:
 epsilon_target  metric_value  eps_vote  p_flip  sigma_equiv pos_rr/clean  pos_recall
            0.5      0.472500   0.00451  0.4989        701.3      197/250        0.42
            1.0      0.544315   0.00884  0.4978        357.7      195/250        0.42
            2.0      0.481538   0.01704  0.4957        185.6      306/250        0.64
            4.0      0.495531   0.03195  0.4920         99.0      306/250        0.64
            8.0      0.484729   0.05758  0.4856         54.9      305/250        0.64

rerun-fidelity replication (teachers retrained once, gaussian cells vs committed grid): {'compared': 12, 'max_abs_auc_delta': 0.0, 'within_1e-6': True}


## Closing task table

| Finding | Status |
|---|---|
| CNN-S + FedProx sanity gate | 0.645→0.690 AUROC r1–r10 — in/above logreg band 0.619 |
| P1 record DP, round-10 band | **utility-flat ε∈[0.5,16]** (all plateau 0.655–0.657); off-arm 0.606 (clip regularization) |
| MIA audit | no detectable advantage at any ε *including off* (attack AUC ≈ 0.500); lower-bound caveat (no shadow models) |
| P2 SecAgg+ | analytic table at CNN wire sizes; TS deployment probe cross-reference (stage-1 wedge recorded there) |
| P3 FedCT | nonviable at bench scale (paid cells NaN); clean ceiling 0.595 AUROC |
| P4 DDG hybrid | proofs verified all rounds; off 0.649 beats P1's off-arm |
| Seed dispersion | 0.579–0.636 (headline ε subset, seeds 43–46); seed-46 worst clinic = 8-row fold artifact |
| P1 per-layer clip | plateau preserved (paid 0.642-0.668 vs global 0.657 at eps>=0.5); filter-health traces per layer+round; MIA clean; cost sqrt(5)x noise for identical composed eps |
| P3 randomized response | partial survival vs Gaussian votes (0.47-0.54 vs ceiling 0.595) BUT rare-positive erosion (pos recall 0.42-0.64); rerun replication verdict max |delta AUROC| = 0.0 |
| User-level DP | **not runnable** — HAM10000 row-permutation unrecoverable (documented limitation) |

Every matrix row's `aux` carries the audit qualifiers (MIA ratios, proof flags, scope notes).